# PEFT 进阶操作

## 1、自定义模型适配

In [1]:
import torch
from torch import nn
from peft import LoraConfig, get_peft_model, PeftModel

In [2]:
net1 = nn.Sequential(nn.Linear(10, 10), nn.ReLU(), nn.Linear(10, 2))
net1

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [3]:
for name, param in net1.named_parameters():
    print(name)

0.weight
0.bias
2.weight
2.bias


In [4]:
# 必须指定 target_modules
config = LoraConfig(target_modules=["0"])

In [5]:
model1 = get_peft_model(net1, config)

In [6]:
model1

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=2, bias=True)
    )
  )
)

## 2、多适配器加载与切换

In [7]:
net2 = nn.Sequential(nn.Linear(10, 10), nn.ReLU(), nn.Linear(10, 2))
net2

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [8]:
config1 = LoraConfig(target_modules=["0"])
model2 = get_peft_model(net2, config1)
model2.save_pretrained("./loraA")

In [9]:
!ls -al ./loraA

total 24
drwxr-xr-x 2 root root 4096 Nov  1 06:07 .
drwxr-xr-x 1 root root 4096 Nov  1 06:08 ..
-rw-r--r-- 1 root root  898 Nov  1 06:13 adapter_config.json
-rw-r--r-- 1 root root  864 Nov  1 06:13 adapter_model.safetensors
-rw-r--r-- 1 root root 5074 Nov  1 06:13 README.md


In [10]:
config2 = LoraConfig(target_modules=["2"])
model2 = get_peft_model(net2, config2)
model2.save_pretrained("./loraB")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [11]:
!ls -al ./loraB

total 24
drwxr-xr-x 2 root root 4096 Nov  1 06:08 .
drwxr-xr-x 1 root root 4096 Nov  1 06:08 ..
-rw-r--r-- 1 root root  898 Nov  1 06:13 adapter_config.json
-rw-r--r-- 1 root root 1432 Nov  1 06:13 adapter_model.safetensors
-rw-r--r-- 1 root root 5074 Nov  1 06:13 README.md


In [12]:
net2 = nn.Sequential(nn.Linear(10, 10), nn.ReLU(), nn.Linear(10, 2))
net2

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [13]:
model2 = PeftModel.from_pretrained(net2, model_id="./loraA/", adapter_name="loraA")
model2

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (loraA): Identity()
        )
        (lora_A): ModuleDict(
          (loraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=2, bias=True)
    )
  )
)

In [14]:
model2.load_adapter("./loraB/", adapter_name="loraB")
model2

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (loraA): Identity()
        )
        (lora_A): ModuleDict(
          (loraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=2, bias=True)
        (lora_dropout): ModuleDict(
          (loraB): Identity()
        )
        (lora_A): ModuleDict(
          (loraB): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraB): Linear(in_features=8, out_features=2, bias=False)
   

In [15]:
model2.active_adapter

'loraA'

In [16]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-1.4719, -0.5307]], grad_fn=<AddmmBackward0>)

In [17]:
for name, param in model2.named_parameters():
    print(name, param)

base_model.model.0.base_layer.weight Parameter containing:
tensor([[ 0.0319, -0.1756,  0.0309, -0.1181,  0.3045, -0.2983,  0.2835,  0.0930,
          0.2561,  0.0600],
        [ 0.2282, -0.0078,  0.2727, -0.2404,  0.0842,  0.1049, -0.1852,  0.2103,
          0.0832,  0.0736],
        [-0.1806,  0.1612, -0.1316, -0.2027,  0.1926, -0.2504,  0.2975, -0.1856,
         -0.0705,  0.0158],
        [ 0.3147, -0.1706, -0.0674,  0.1621,  0.2859,  0.0387,  0.2500, -0.1249,
         -0.2550,  0.2561],
        [ 0.2245, -0.3109,  0.2227, -0.2236, -0.2619,  0.2797, -0.3018, -0.0425,
         -0.2762, -0.1612],
        [-0.2619, -0.2814, -0.2369, -0.1099, -0.3121,  0.2505, -0.2248,  0.1554,
         -0.1401,  0.3102],
        [-0.1261,  0.1205,  0.2316,  0.0366, -0.2974, -0.2657,  0.2935, -0.1126,
          0.2277,  0.1964],
        [-0.1340,  0.1424, -0.0724,  0.0642, -0.3135, -0.0391,  0.2264,  0.0084,
         -0.2250,  0.2468],
        [ 0.1998,  0.0680,  0.3088,  0.1675, -0.1855,  0.0323,  0.083

In [18]:
for name, param in model2.named_parameters():
    if name in [
        "base_model.model.0.lora_A.loraA.weight",
        "base_model.model.0.lora_B.loraA.weight",
    ]:
        param.data = torch.ones_like(param)

In [19]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-355.0695, -171.7003]], grad_fn=<AddmmBackward0>)

In [20]:
model2.set_adapter("loraB")

In [21]:
model2.active_adapter

'loraB'

In [22]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-1.4719, -0.5307]], grad_fn=<AddBackward0>)

## 3、禁用适配器

In [23]:
model2.set_adapter("loraA")

In [24]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-355.0695, -171.7003]], grad_fn=<AddmmBackward0>)

In [25]:
with model2.disable_adapter():
    print(model2(torch.arange(0, 10).view(1, 10).float()))

tensor([[-1.4719, -0.5307]])
